In [ ]:
#imports
import os
import zipfile
from datetime import datetime
import requests
import pandas as pd
from google.transit import gtfs_realtime_pb2

In [ ]:
#setup
BASE_DIR = "mta_project_data"
STATIC_DIR = f"{BASE_DIR}/gtfs_static"
REALTIME_DIR = f"{BASE_DIR}/gtfs_realtime_snapshots"

os.makedirs(STATIC_DIR, exist_ok=True)
os.makedirs(REALTIME_DIR, exist_ok=True)

# 1. Static GTFS Subway Dataset
STATIC_GTFS_URL = "https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip"

# 2. Realtime GTFS feeds
REALTIME_FEEDS = {
    "1234567S": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs",
    "ACE": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-ace",
    "BDFM": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-bdfm",
    "G": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-g",
    "JZ": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-jz",
    "NQRW": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-nqrw",
    "L": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-l"
}

In [11]:
def download_static_gtfs():
    zip_path = f"{BASE_DIR}/gtfs_subway.zip"

    #print("Downloading static GTFS...")
    response = requests.get(STATIC_GTFS_URL)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(STATIC_DIR)

    #print("Static GTFS downloaded and extracted.")


def load_static_tables():
    #print("\nStatic GTFS tables:")

    for file in os.listdir(STATIC_DIR):
        if file.endswith(".txt"):
            path = f"{STATIC_DIR}/{file}"
            df = pd.read_csv(path)
            #print(f"\n{file}")
            #print(df.head())
            #print("Columns:", list(df.columns))


def collect_realtime_snapshot():
    all_rows = []

    for feed_name, url in REALTIME_FEEDS.items():
        #print(f"Collecting realtime feed: {feed_name}")

        response = requests.get(url)
        response.raise_for_status()

        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)

        for entity in feed.entity:
            if entity.HasField("vehicle"):
                vehicle = entity.vehicle

                all_rows.append({
                    "feed_name": feed_name,
                    "entity_id": entity.id,
                    "trip_id": vehicle.trip.trip_id,
                    "route_id": vehicle.trip.route_id,
                    "direction_id": vehicle.trip.direction_id,
                    "start_time": vehicle.trip.start_time,
                    "start_date": vehicle.trip.start_date,
                    "schedule_relationship": vehicle.trip.schedule_relationship,
                    "current_stop_sequence": vehicle.current_stop_sequence,
                    "current_status": vehicle.current_status,
                    "stop_id": vehicle.stop_id,
                    "timestamp_unix": vehicle.timestamp,
                    "timestamp_readable": datetime.fromtimestamp(vehicle.timestamp) if vehicle.timestamp else None,
                    "vehicle_id": vehicle.vehicle.id,
                    "vehicle_label": vehicle.vehicle.label,
                    "latitude": vehicle.position.latitude,
                    "longitude": vehicle.position.longitude,
                    "bearing": vehicle.position.bearing,
                    "odometer": vehicle.position.odometer,
                    "speed": vehicle.position.speed,
                    "congestion_level": vehicle.congestion_level,
                    "occupancy_status": vehicle.occupancy_status
                })

    df = pd.DataFrame(all_rows)

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"{REALTIME_DIR}/mta_realtime_snapshot_{now}.csv"

    df.to_csv(output_path, index=False)

    #print("\nRealtime snapshot saved:")
    #print(output_path)

    #print("\nRealtime table preview:")
    #print(df.head())

    #print("\nColumns:")
    #print(list(df.columns))
    print(df.isna().sum())

    return df


if __name__ == "__main__":
    download_static_gtfs()
    load_static_tables()
    realtime_df = collect_realtime_snapshot()

feed_name                0
entity_id                0
trip_id                  0
route_id                 0
direction_id             0
start_time               0
start_date               0
schedule_relationship    0
current_stop_sequence    0
current_status           0
stop_id                  0
timestamp_unix           0
timestamp_readable       0
vehicle_id               0
vehicle_label            0
latitude                 0
longitude                0
bearing                  0
odometer                 0
speed                    0
congestion_level         0
occupancy_status         0
dtype: int64
